<a href="https://colab.research.google.com/github/ayoushbirjani4-spec/fiyrank-firstlab/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayoushbirjani4-spec/fiyrank-firstlab/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows |", df.shape[1], "columns")

30000 rows | 44 columns


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Refresh / Content Opportunity Scoring.** The actual decision an editor faces is "which
pages should I review first this week" — that's fundamentally a **ranking/scoring** problem, not
a bare classification one. An editor doesn't want a flat yes/no on 30,000 pages; they want an
ordered queue they can work top-down until they run out of hours.

Underneath the ranking, though, the score itself is built from a **classification** signal —
`is_declining_label` (down/not-down) — because "probability this page is declining" is what I
rank by. So: **scoring/ranking, backed by a binary classifier's probability output**, evaluated
with a ranking-appropriate metric (Precision@K), not a classification-appropriate one (plain
accuracy would reward being right about the 85% of pages that *aren't* declining and tell me
nothing about whether the top of my queue is good).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label`, defined the same way notebook 02 already establishes it —
`trend_direction.str.lower() == "down"`. This is an **observed outcome**, not an invented rule:
`trend_direction` is computed by the warehouse from real impressions comparing `last_30d` vs
`prev_30d`, not something I'm defining after the fact to make my model look good.

**Why it's a proxy, not a perfect target:** "declining" here means *impressions* dropped,
which is a proxy for the real thing an editor cares about — whether the page is *losing
business value* (which could also mean falling conversions, losing to a competitor on
a specific query, or going stale content-wise even while impressions hold steady). I'm using
the observable proxy because the ideal target isn't measured in this data at all.

**The leakage trap to avoid (from notebook 02):** `trend_pct` is the exact percentage this
label is thresholded from — feeding it back in as a feature would just teach a model to
re-discover its own label. Same reasoning as the w03 data contract.

In [2]:
print(df["trend_direction"].value_counts(dropna=False))
print()

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("declining rate:", round(df["is_declining_label"].mean(), 3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K**, specifically **Precision@50** — of the top 50 pages my score ranks highest, what
fraction are actually declining. This matches the real decision: an editor has limited hours and
works a fixed-size queue, not the whole dataset, so a metric that scores the *entire* ranking
(like full-dataset AUC) rewards being right about pages nobody will ever look at. Precision@K
only rewards being right about the pages that actually get acted on.

**What "good" means here, concretely:** the baseline declining rate across all pages (computed
below) is the number a random ranking would get at any K. Precision@50 needs to beat that base
rate by a meaningful margin to justify the ranking work over just picking pages at random.

In [3]:
base_rate = df["is_declining_label"].mean()
print(f"Base declining rate (what a random top-50 would score): {base_rate:.3f}")
print("A ranking is only worth defending if Precision@50 clears this by a real margin.")

Base declining rate (what a random top-50 would score): 0.542
A ranking is only worth defending if Precision@50 clears this by a real margin.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (`content_id`), for one client (`client_id`)** — a static snapshot
per page, not a time series. There's no `report_date` in this starter CSV; `impressions_90d`,
`trend_direction`, etc. are already pre-aggregated per page by the time this file was released.
That's a real difference from the w03 data contract's daily-fact grain — worth naming so I don't
accidentally treat this like the same table.

In [4]:
print("Rows:", len(df), "| Unique content_id:", df["content_id"].nunique(),
      "| Unique client_id:", df["client_id"].nunique())

cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "trend_direction", "is_declining_label"]
df[cols].head(10)

Rows: 30000 | Unique content_id: 30000 | Unique client_id: 32


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single-signal rule exists and is cheap: "stale (not updated in 180+ days) AND still visible
(500+ impressions)". Notebook 02 already built exactly this hand rule and scored it. It's not
useless — at small K it's competitive.

**Where it breaks down:** the rule only combines two signals with a fixed threshold. Real decline
patterns involve `avg_position`, `ctr`, `engagement_rate`, `word_count`, `content_age_days`, and
`ai_traffic_pct` interacting in ways that don't reduce to one clean AND. A page can be fresh but
still declining (competitor outranked it), or stale but stable (evergreen content with low
volatility) — the fixed rule can't distinguish these, but a model with more signals can weigh
them jointly. That's the actual argument for ML: not that the rule is *wrong*, but that it's
throwing away signal a slightly more flexible model can use — which is exactly what notebook 02's
depth-2 tree comparison already demonstrates empirically, not just asserts.

**Where ML does NOT help:** it's not needed to invent the label (that's still `trend_direction`,
an observed rule from real data, not model output), and a plain dashboard would suffice if an
editor just wanted to *browse* trends rather than get a prioritized action queue.

In [5]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}  (base rate: {base_rate:.3f})")

print()
print("This is the number a model needs to beat to justify the extra complexity —")
print("full model-vs-rule comparison is w02_your_first_readable_model.ipynb, not repeated here.")

Hand rule Precision@20: 0.900  (base rate: 0.542)
Hand rule Precision@50: 0.680  (base rate: 0.542)

This is the number a model needs to beat to justify the extra complexity —
full model-vs-rule comparison is w02_your_first_readable_model.ipynb, not repeated here.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.